# Đào tạo Keyword Spotting với mô hình Honk (Res15)

Notebook này đã được cập nhật toàn bộ các bản vá lỗi tự động để tương thích 100% với môi trường Kaggle hiện tại. Các lỗi liên quan đến `librosa`, `pyaudio`, `pcen` và `chainmap` đều được xử lý ngầm ở Bước 2 & 3.

### Cách chạy:
1. Đảm bảo ở mục **Accelerator** bên phải đã chọn **GPU T4x2** hoặc **GPU P100**.
2. Bấm **Run All** để chạy tuần tự từ trên xuống dưới.

In [1]:
# Bước 1: Tải mã nguồn dự án Honk
!git clone https://github.com/castorini/honk.git
%cd honk

Cloning into 'honk'...
remote: Enumerating objects: 781, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 781 (delta 0), reused 0 (delta 0), pack-reused 777 (from 1)
Receiving objects: 100% (781/781), 5.93 MiB | 17.25 MiB/s, done.
Resolving deltas: 100% (450/450), done.
/kaggle/working/honk


In [2]:
# Bước 2: Cập nhật thư viện hệ thống và môi trường
!sed -i 's/==.*//g' requirements.txt
!sed -i 's/>=.*//g' requirements.txt
!sed -i 's/<=.*//g' requirements.txt

!apt-get update > /dev/null
!apt-get install -y portaudio19-dev > /dev/null
!pip install pyaudio git+https://github.com/daemon/pytorch-pcen > /dev/null
!pip install -r requirements.txt > /dev/null
print("Hoàn tất cài đặt thư viện!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
  Running command git clone --filter=blob:none --quiet https://github.com/daemon/pytorch-pcen /tmp/pip-req-build-inzf1s0k
  Running command git clone --filter=blob:none --quiet https://github.com/daemon/pytorch-pcen /tmp/pip-req-build-feqq86hp
Hoàn tất cài đặt thư viện!


In [3]:
# Bước 3: Vá các hàm cũ của thư viện Python (Monkey-patch)
with open('utils/model.py', 'r') as f:
    content = f.read()
with open('utils/model.py', 'w') as f:
    f.write(content.replace('from chainmap import ChainMap', 'from collections import ChainMap'))

with open('utils/manage_audio.py', 'r') as f:
    content = f.read()

patch = """
import numpy as np
if not hasattr(librosa.filters, 'dct'):
    def _dct(n_filters, n_input):
        basis = np.empty((n_filters, n_input))
        basis[0, :] = 1.0 / np.sqrt(n_input)
        samples = np.arange(1, 2 * n_input, 2) * np.pi / (2.0 * n_input)
        for i in range(1, n_filters):
            basis[i, :] = np.cos(i * samples) * np.sqrt(2.0 / n_input)
        return basis
    librosa.filters.dct = _dct
"""
with open('utils/manage_audio.py', 'w') as f:
    f.write(content.replace('import librosa\n', 'import librosa\n' + patch))
    
print("Đã vá code cũ thành công!")

Đã vá code cũ thành công!


In [4]:
# Bước 4: Tải Google Speech Commands Dataset và các Mô hình Pre-trained
!chmod +x fetch_data.sh
!./fetch_data.sh

--2026-09-12 18:33:15--  https://github.com/castorini/honk-data/archive/master.zip
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/castorini/honk-data/zip/refs/heads/master [following]
--2026-09-12 18:33:16--  https://codeload.github.com/castorini/honk-data/zip/refs/heads/master
Resolving codeload.github.com (codeload.github.com)... 20.27.177.114
Connecting to codeload.github.com (codeload.github.com)|20.27.177.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘data.zip’

data.zip                [      <=>           ]  28.34M  25.1MB/s    in 1.1s    

2026-09-12 18:33:17 (25.1 MB/s) - ‘data.zip’ saved [29714224]

--2026-09-12 18:33:17--  https://github.com/castorini/honk-models/archive/master.zip
Resolving github.com (github.com)... 20.27.177.113
Connecting t

In [5]:
!wget http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz
!mkdir speech_dataset
!tar -xf speech_commands_v0.02.tar.gz -C speech_dataset

--2026-09-12 18:33:20--  http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz
Resolving download.tensorflow.org (download.tensorflow.org)... 64.233.188.207, 108.177.97.207, 74.125.23.207, ...
Connecting to download.tensorflow.org (download.tensorflow.org)|64.233.188.207|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2428923189 (2.3G) [application/gzip]
Saving to: ‘speech_commands_v0.02.tar.gz’

speech_commands_v0. 100%[===================>]   2.26G  21.0MB/s    in 1m 47s  

2026-09-12 18:35:07 (21.7 MB/s) - ‘speech_commands_v0.02.tar.gz’ saved [2428923189/2428923189]



In [6]:
# Tự động chèn từ khóa y= vào hàm melspectrogram cho tương thích Librosa 0.11+
with open('utils/manage_audio.py', 'r') as f:
    c = f.read()

c = c.replace('librosa.feature.melspectrogram(\n            data,', 'librosa.feature.melspectrogram(\n            y=data,')
c = c.replace('librosa.feature.melspectrogram(audio_data', 'librosa.feature.melspectrogram(y=audio_data')

with open('utils/manage_audio.py', 'w') as f:
    f.write(c)

print("Đã sửa lỗi tham số melspectrogram thành công!")

Đã sửa lỗi tham số melspectrogram thành công!


In [7]:
# Bước 5: Bắt đầu quá trình Huấn luyện (End-to-End Training)
# Sử dụng mô hình mạng Residual Network 15 lớp (res15)
!python -m utils.train --data_folder speech_dataset --model res15 --wanted_words yes no up down left right on off stop go --dev_every 1 --n_labels 12 --n_epochs 26 --weight_decay 0.00001 --lr 0.1 0.01 0.001 --schedule 3000 6000

train step #1 accuracy: 0.046875, loss: 2.4854817390441895       
train step #2 accuracy: 0.078125, loss: 2.4950754642486572       
train step #3 accuracy: 0.09375, loss: 2.5160093307495117       
train step #4 accuracy: 0.0625, loss: 2.527837038040161        
train step #5 accuracy: 0.21875, loss: 2.467341423034668        
train step #6 accuracy: 0.125, loss: 2.472942352294922        
train step #7 accuracy: 0.1875, loss: 2.4638900756835938       
train step #8 accuracy: 0.15625, loss: 2.4460136890411377       
train step #9 accuracy: 0.171875, loss: 2.4567551612854004       
train step #10 accuracy: 0.15625, loss: 2.4235544204711914       
train step #11 accuracy: 0.21875, loss: 2.4245104789733887       
train step #12 accuracy: 0.125, loss: 2.406547784805298        
train step #13 accuracy: 0.078125, loss: 2.449979543685913        
train step #14 accuracy: 0.140625, loss: 2.392972946166992        
train step #15 accuracy: 0.171875, loss: 2.377756118774414        
train step #16 accu